In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mabs_funcs as mf
import mfunc_winds as mf2

import matplotlib.colors as colors
import matplotlib.cm as cm

from matplotlib.colors import SymLogNorm
from matplotlib.colors import LogNorm

from scipy.stats import binned_statistic_2d

In [ ]:
Uavg = mf.define_rings()
Uavg_phy = mf.du_phy * Uavg
rdata, rcount = mf.CalCR(Uavg)

In [ ]:
rdata

In [ ]:
# file_loc_v1='/scratch/sukhdeep/project_MABS/on_data/4_fbe/results/v1'
# file_loc_v2='/scratch/sukhdeep/project_MABS/on_data/4_fbe/results/v2'

In [ ]:
file_loc_v2='/scratch/sukhdeep/project_MABS/on_data/4_fbe/results/LR_pol/mwa_data'

In [ ]:
rcount2, rdata2 = mf.read_rings(file_loc_v2)
Ntri = np.round(rdata2.T[3]/(1176*1176))

In [ ]:
## also defined in mf2 similarly
Nnc  = 2*(mf.nc10+1 - mf.n_nancut)-1 
Nnc_hf = Nnc//2+1
L   = Nnc * mf.dnu_mwa * mf.rc_prime
rc4 = mf.rc**4
Lsq = L**2
dkll = 2*np.pi/L

In [ ]:
r3,r2,r1 = 15,15,15

mabs_res1 = mf2.mabs_win3(r1,r2,r3,file_loc_v2, 'mwa' ,1)

Nnc  = mabs_res1.shape[1]   # 2*(mf.nc10+1 - mf.n_nancut)-1 
Nnc_hf = Nnc//2+1
plt.imshow(mabs_res1,origin='lower',extent=[-Nnc_hf*mf.dnu_mwa,Nnc_hf*mf.dnu_mwa,-Nnc_hf*mf.dnu_mwa,Nnc_hf*mf.dnu_mwa],cmap='coolwarm') #,vmin=0,vmax=0.03
plt.colorbar(location='top')

In [ ]:
mabs_res1 = mf2.mabs_win(r3,r2,r1,file_loc_v2, 'mwa' ,1)
Nnc  = mabs_res1.shape[1]   # 2*(mf.nc10+1 - mf.n_nancut)-1 
Nnc_hf = Nnc//2+1
plt.imshow(mabs_res1,origin='lower',extent=[-Nnc_hf*mf.dnu_mwa,Nnc_hf*mf.dnu_mwa,0,Nnc_hf*mf.dnu_mwa],cmap='coolwarm') #,vmin=0,vmax=0.03
plt.colorbar(location='top')

In [ ]:
r1,r2,r3=16,16,16
mabs_res1 = mf2.mabs(r3,r2,r1,file_loc_v2, 'mwa' ,1)

Nnc  = mabs_res1.shape[1]   # 2*(mf.nc10+1 - mf.n_nancut)-1 
Nnc_hf = Nnc//2+1
plt.imshow(mabs_res1,origin='lower',extent=[-Nnc_hf*mf.dnu_mwa,Nnc_hf*mf.dnu_mwa,0,Nnc_hf*mf.dnu_mwa],cmap='coolwarm') #,vmin=0,vmax=0.03
plt.colorbar(location='top')

# Wedge

In [ ]:
largest_ring = 22

kp_vals = np.array([mf.kp(Uavg_phy,i1) for i1 in range(largest_ring)])

midpoints = (kp_vals[1:] + kp_vals[:-1]) / 2
first_edge = kp_vals[0] - (midpoints[0] - kp_vals[0])
last_edge  = kp_vals[-1] + (kp_vals[-1] - midpoints[-1])
x_edges = np.concatenate(([first_edge], midpoints, [last_edge]))
# print(x_edges)


kll_vals = np.linspace( -dkll*(Nnc_hf-1), dkll*(Nnc_hf-1), Nnc )[Nnc_hf-1:] 
internal_edges = 0.5 * (kll_vals[1:] + kll_vals[:-1])  # shape: (len(x_vals)-1)

first_edge = kll_vals[0] - 0.5 * (kll_vals[1] - kll_vals[0])
last_edge  = kll_vals[-1] + 0.5 * (kll_vals[-1] - kll_vals[-2])
y_edges = np.concatenate(([first_edge], internal_edges, [last_edge]))  # shape: (len(x_vals)+1,)
# print(y_edges)

In [ ]:
# results = []
# jj = 0#Nnc_hf
# for i in range(largest_ring):
#     res_1    = mf2.cylbs(i,i,0,file_loc_v2,bin_flag=False,n_bins=30)
#     res_k2pf = np.abs(res_1[jj])[Nnc_hf-1:]
#     results.append(res_k2pf)
    
# results_2d = np.array(results)



In [ ]:
results = []
jj = 0#Nnc_hf
for i in range(largest_ring):
    res_k2pf = np.zeros(Nnc_hf)
    # res_1    = mf2.cylbs(i,i,0,file_loc_v2,bin_flag=False,n_bins=30)
    res_1    = mf2.cylbs_win(i,i,i,file_loc_v2, 'mwa' ,1)
    for ia in range(Nnc_hf-1,Nnc):
        res_k2pf[ia-Nnc_hf+1] = np.abs(res_1[-ia-1][ia])
        # res_k2pf[ia-Nnc_hf+1] = np.abs(res_1[-ia-1][Nnc_hf-1])  #### gives good wedge
        # res_k2pf[ia-Nnc_hf+1] = res_1[-ia-1][Nnc_hf-1]
    results.append(res_k2pf)
    
results_2d = np.array(results)

In [ ]:
kll_H = np.zeros(largest_ring)
kp_H  = np.zeros(largest_ring)
for i in range(largest_ring):
    kll_H[i] = mf.rc*mf.kp(Uavg_phy,i)/(mf.rc_prime*mf.nu0*1e-6)
    kp_H[i]  = mf.kp(Uavg_phy,i)

In [ ]:
abs_flag  = True
rem_kp = 1
rem_kpend=6

linthresh = 1e0
linscale  = 1e0
res_plt_2   = results_2d if abs_flag==False else np.abs(results_2d)
res_plt   = res_plt_2.T[:,rem_kp:]
vmax      =  np.max(res_plt)/1e0 # 1e18 #
vmin      =  np.min(res_plt)*1e0 #5e21#5e11#np.min(res_plt) # 1e11#


# Ny, Nx = res_plt.shape
# x_edges = np.linspace(0, dkll*(Nnc_hf-1), Nx + 1)
# y_edges = np.concatenate(([first_edge], midpoints, [last_edge]))
y_edges[0]= 1e-3
x_edges1 = x_edges[rem_kp:]
# Create figure and axis
fig, ax1 = plt.subplots(1, 1, figsize=(4, 4))

# Use SymLogNorm for symlog color normalization
norm = LogNorm(vmin=vmin, vmax=vmax) #if abs_flag==True else SymLogNorm(linthresh=linthresh, linscale=linscale, vmin=vmin, vmax=vmax)
# norm = SymLogNorm(linthresh=linthresh, linscale=linscale, vmin=vmin, vmax=vmax)

# Plot using pcolormesh
mesh = ax1.pcolormesh(x_edges1, y_edges, res_plt, cmap='jet', norm=norm, shading='auto')

# Set symlog scales for x and y axes
# ax1.set_xscale('symlog', linthresh=0.01)
# ax1.set_yscale('symlog', linthresh=0.01)

ax1.set_xscale('log')
ax1.set_yscale('log')

# Axis labels and title
ax1.set_title(r"Estimated")
ax1.set_ylabel(r'$k_{\parallel}$')
ax1.set_xlabel(r'$k_{\perp}$')

ax1.plot(kp_H[rem_kp:],kll_H[rem_kp:],'--',color='k',lw=2)


# Axis limits
# ax1.set_xlim(-0.2, 0.2)
# ax1.set_ylim(-0.2, 0.2)

# Add horizontal colorbar
cbar_ax = fig.add_axes([0.1, 1, .8, 0.05])
cbar = fig.colorbar(mesh, cax=cbar_ax, orientation='horizontal', location='top')
cbar.set_label(r'$B(k_{1\perp},k_{2\perp},k_{3\perp},k_{1\parallel},k_{2\parallel})$')

In [ ]:
xrange = np.linspace( 0, dkll*(Nnc_hf-1), Nnc_hf )

i1 = 5
res_plot1 = (results_2d[i1])
i2 = 10
res_plot2 = (results_2d[i2])
i3 = 15
res_plot3 =(results_2d[i3])
# np.abs

# plt.plot(xrange,res_1[i],'.')
plt.plot(xrange,res_plot1,'--',label='kll=%f'%xrange[i1])
plt.plot(xrange,res_plot2,'--',label='kll=%f'%xrange[i2])
plt.plot(xrange,res_plot3,'--',label='kll=%f'%xrange[i3])
# plt.vlines([wedge],1e14,1e17)
# plt.xlim(0.01,1)
# plt.xscale('log')
plt.yscale('symlog',linthresh=1e16)
# plt.yscale('log')

# plt.legend(fontsize=12)

In [ ]:
xrange = np.linspace( 0, dkll*(Nnc_hf-1), Nnc_hf )

cmap = cm.get_cmap('jet')
colors = cmap(np.linspace(0, 1, largest_ring))

i1 = 4
# res_plot1 = np.abs(results_2d[i1])
res_plot1 = results_2d[i1]
plt.plot(xrange,res_plot1,'-', color=colors[i1],label=r'$k_\perp=%f$'%mf.kp(Uavg_phy,i1))

plt.vlines(.5,1e10,1e17,'k')
# plt.xlim(0.01,1)
# plt.xscale('log')
plt.yscale('symlog',linthresh=1e1)
# plt.yscale('log')

plt.legend(fontsize=10)
plt.xlabel('$k_\parallel$',fontsize=15)
plt.ylabel('$B(k_\perp,k_\parallel)$',fontsize=15)

In [ ]:
xrange = np.linspace( 0, dkll*(Nnc_hf-1), Nnc_hf )

cmap = cm.get_cmap('jet')
colors = cmap(np.linspace(0, 1, largest_ring))

for i1 in np.arange(0,largest_ring,3):
    res_plot1 = np.abs(results_2d[i1])
    plt.plot(xrange,res_plot1,'-', color=colors[i1],label=r'$k_\perp=%f$'%mf.kp(Uavg_phy,i1))

# plt.vlines([wedge],1e14,1e17)
# plt.xlim(0.01,1)
# plt.xscale('log')
# plt.yscale('symlog',linthresh=1e16)
plt.yscale('log')

plt.legend(fontsize=10)
plt.xlabel('$k_\parallel$',fontsize=15)
plt.ylabel('$B(k_\perp,k_\parallel)~[Mpc^6~mK^3]$',fontsize=15)

In [ ]:
xrange = np.linspace( 0, dkll*(Nnc_hf-1), Nnc_hf )

cmap = cm.get_cmap('jet')
colors = cmap(np.linspace(0, 1, 16))

# for i1 in np.arange(0,1):
i1 = 4
res_plot1 = np.abs(results_2d[i1])
plt.plot(xrange,res_plot1,'-', color='navy',label=r'$k_\perp=%f$'%mf.kp(Uavg_phy,i1))
    # plt.vlines([.29,.58,.87],1e17,1e20)

# plt.vlines([wedge],1e14,1e17)
# plt.xlim(0.01,1)
# plt.xscale('log')
plt.yscale('symlog',linthresh=1e16)
plt.yscale('log')

plt.legend(fontsize=10)
plt.xlabel('$k_\parallel~[Mpc^{-1}]$',fontsize=15)
plt.ylabel('$|B(k_\perp,k_\parallel)|~[Mpc^6~mK^3]$',fontsize=15)

# 3DBS

In [ ]:
rcount2, rdata2 = mf.read_rings(file_loc_v2)
Ntri = np.round(rdata2.T[3]/(1176*1176))

In [ ]:
k1_bins = 12
t_bins  = 5
mu_bins = 5


n_shapes = 669
Nnc  = 2*(mf.nc10+1 - mf.n_nancut)-1   ##mf.nc10+1
LL   = mf.dnu_mwa * mf.rc_prime
L    = Nnc * LL
print(mf.dnu_mwa,LL,L)
dkll = 2 * np.pi / L

indices = np.fft.fftshift(np.fft.fftfreq(Nnc, d=1.0/Nnc))
i_vals, j_vals = np.meshgrid(indices, indices, indexing='ij')
k2_para = dkll * i_vals
k3_para = dkll * j_vals
k1_para = - k3_para - k2_para

k1_edges = np.logspace(np.log10(0.002), np.log10(4.624), k1_bins + 1)
t_edges  = np.linspace(.5, 1., t_bins + 1)
mu_edges = np.linspace(.5, 1., mu_bins + 1)
bin_edges = [k1_edges, t_edges, mu_edges]

# file_loc='/scratch/sukhdeep/project_MABS/3VC_B/4_fbe/results/final'
# file_loc='/scratch/sukhdeep/project_MABS/3VC_B/4_fbe/results/flagged/coll'

In [ ]:
mask = np.ones((Nnc,Nnc))

In [ ]:
start_shape = 0
Rlzn = 1
dnu_file_loc = file_loc_v2 #file_loc_NF

bs,num = mf2.get_binned_3DBS(bin_edges,start_shape, n_shapes, rdata2, Uavg_phy, k2_para, k3_para, dnu_file_loc, mask, Rlzn)


In [ ]:
# np.savez("bs_mwa_win_kp22rings_allkll_12X5X5",bs,num)

In [ ]:
res_1    = mf2.cylbs_win(17,17,17,file_loc_v2)

plt.imshow(res_1,origin='lower',extent=[-dkll*(Nnc_hf-1),dkll*(Nnc_hf-1), -dkll*(Nnc_hf-1), dkll*(Nnc_hf-1)])
plt.ylim(-.2,.2)
plt.xlim(-.2,.2)

In [ ]:
k2_para.shape
plt.imshow(k1_para,origin='lower',extent=[-dkll*(Nnc_hf-1),dkll*(Nnc_hf-1), -dkll*(Nnc_hf-1), dkll*(Nnc_hf-1)])
# plt.ylim(-.2,.2)
# plt.xlim(-.2,.2)

In [ ]:
kp_vals = np.array([mf.kp(Uavg_phy,i1) for i1 in range(largest_ring)])

# FG Avoid

In [ ]:
k1_bins = 10
t_bins  = 5
mu_bins = 5


n_shapes = 365 #669
Nnc  = 2*(mf.nc10+1 - mf.n_nancut)-1   ##mf.nc10+1
LL   = mf.dnu_mwa * mf.rc_prime
L    = Nnc * LL
print(mf.dnu_mwa,LL,L)
dkll = 2 * np.pi / L

indices = np.fft.fftshift(np.fft.fftfreq(Nnc, d=1.0/Nnc))
i_vals, j_vals = np.meshgrid(indices, indices, indexing='ij')
k2_para = dkll * i_vals
k3_para = dkll * j_vals
k1_para = - k3_para - k2_para

k1_edges = np.logspace(np.log10(0.5), np.log10(4.624), k1_bins + 1)
t_edges  = np.linspace(.5, 1., t_bins + 1)
mu_edges = np.linspace(.5, 1., mu_bins + 1)
bin_edges = [k1_edges, t_edges, mu_edges]

# file_loc='/scratch/sukhdeep/project_MABS/3VC_B/4_fbe/results/final'
# file_loc='/scratch/sukhdeep/project_MABS/3VC_B/4_fbe/results/flagged/coll'

In [ ]:
### Foreground avoidance ###

## mask on kparallel, kperp plane
# kp_vals = np.array([mf.kp(Uavg_phy,i1) for i1 in range(22)])
# kll_vals = np.linspace( -dkll*(Nnc_hf-1), dkll*(Nnc_hf-1), Nnc )[Nnc_hf-1:] 
# kllmesh, kpmesh = np.meshgrid(kll_vals, kp_vals[1:], indexing='ij')
# kp1s = np.where(kpmesh > 0.08, 0, 1)
# kll1s= np.where(kllmesh<0.5, 0, 1)
# remains = kp1s*kll1s
# mask = remains

### Mask on k1ll, k2ll plane ###
klv = np.linspace( -dkll*(Nnc_hf-1), dkll*(Nnc_hf-1), Nnc )
kl1m, kl2m = np.meshgrid(klv, klv, indexing='ij')
kl3m  = -kl1m - kl2m
kl1m_cut = np.where(np.abs(kl1m) < 0.5, 0, 1)
kl2m_cut = np.where(np.abs(kl2m) < 0.5, 0, 1)
kl3m_cut = np.where(np.abs(kl3m) < 0.5, 0, 1)
remains = kl1m_cut*kl2m_cut*kl3m_cut
mask = remains

In [ ]:
start_shape = 1
Rlzn = 1
dnu_file_loc = file_loc_v2 #file_loc_NF

bs,num = mf2.get_binned_3DBS(bin_edges,start_shape, n_shapes, rdata2, Uavg_phy, k2_para, k3_para, dnu_file_loc, mask, Rlzn)


In [ ]:
# np.savez("test",bs,num)

In [ ]:
# np.savez("bs_mwa_win_kp22rings_allkll_10X5X5_FR_0exandk3llp5",bs,num)

In [ ]:
plt.imshow(mask,origin='lower')